# 06 - Figure 2

Thin caller: shared data-load cells are replaced with `nfip` loader calls;

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
from src import data, config
from src.plotting import *   # shared figure style defaults

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, TwoSlopeNorm, ListedColormap
from matplotlib.patches import Patch
from scipy.cluster.hierarchy import linkage, fcluster

# Parameters

In [ ]:
save = True
input_clustering = '_new' # '_new' or ''
optimal_cluster = 'st_cluster_final_gid' # 'st_cluster_final_gid' or 'st_cluster_3_5_7'

test_case = 'block'
historic_ref = '' # '_Full' '_Discount' or ''
premium_type = '' # '_Full'  '_Discount' or ''

use_reinsurance = True
base_case = ''
if use_reinsurance == False:
    base_case = '_base_case'

# Data Load

## Simulations

In [ ]:
res = data.load_simulation_results(test_case, input_clustering, premium_type,
                                   base_case, historic_ref=historic_ref)
final_balances_df = res['final_balances']
cluster_state_df = res['cluster_state']

## Geospatial

In [ ]:
gdf_states = data.load_states()
gdf_states["STATEFP"] = pd.to_numeric(gdf_states["STATEFP"], errors="coerce").astype("Int64")

## NFIP Policies

In [ ]:
aggregated_risk_policies = data.load_premiums(premium_type)

state_fips_dict = config.FIPS_TO_NAME
# Convert the dictionary to a DataFrame for easy merging
state_df = pd.DataFrame(list(state_fips_dict.items()), columns=['State FIPS', 'State Name'])

state_abbrev_dict = config.FIPS_TO_ABBREV

In [ ]:
state_claims = pd.read_csv('../Local_Data/NFIP_Data/All_Claims_by_Year.csv')
state_claims = state_claims.groupby("State").aggregate({'Total Claim Dollars Paid':'mean','Total Paid Claims':'mean'}).reset_index()
state_claims['State'] = state_claims['State'].str.upper()
state_claims = state_claims.merge(state_df, left_on='State', right_on='State Name', how='left')
state_policies = aggregated_risk_policies.groupby("State").aggregate({
    'Policies in Force':'sum',
    'Total Written Premium + FPF':'sum',
    'Total Annual Payment':'sum'}).reset_index()
state_merged = state_claims.merge(state_policies, left_on='State', right_on='State', how='left')
state_merged['meanLoss'] = state_merged['Total Written Premium + FPF']-state_merged['Total Claim Dollars Paid']

# Merge the aggregated data with the shapefile
gdf_states = gdf_states.merge(state_merged, left_on='GEOID', right_on='State FIPS', how='left')

In [ ]:
# Restrict the mapped domain to the simulated states
gdf_states = gdf_states[~gdf_states['State'].isin(['ALASKA', 'HAWAII'])]

# Plotting

# Benefit and Strain

In [ ]:
# Sum reinsurance used and final balance per state
re_summary = (
    cluster_state_df.groupby("STATEFP")[["re_covered"]]
    .sum()
    .reset_index()
)

balance_summary = (
    final_balances_df.groupby("STATEFP")[["nfip_balance"]]
    .mean()  # or .last() if only using final year
    .reset_index()
)

premium_summary = (
    gdf_states[["STATEFP", "Total Written Premium + FPF", "Policies in Force"]]
    .rename(columns={"Total Written Premium + FPF": "premium","Policies in Force": "no_policies"})
)

# Merge all together
summary_df = (
    re_summary
    .merge(balance_summary, on="STATEFP", how="outer")
    .merge(premium_summary, on="STATEFP", how="left")
)

# Drop states with zero or missing premium
summary_df = summary_df[summary_df["premium"] > 0].copy()
summary_df["premium"] = summary_df["premium"].replace(0, np.nan)

# Derived Metrics
summary_df["Norm_re"] = summary_df["re_covered"] / summary_df["premium"] /1000
summary_df["Norm_balance"] = summary_df["nfip_balance"] / summary_df["premium"] /100

# System-wide benefit
total_balance = summary_df["nfip_balance"].sum()
total_premiums = summary_df["premium"].sum()
epsilon = 1

summary_df["Net_benefit"] = total_balance / summary_df["nfip_balance"].clip(lower=total_balance/10000)
neg_def = total_balance
summary_df["OG_Cont"]= summary_df["premium"] / total_premiums
summary_df["Original_benefit"] = total_premiums / summary_df["premium"]
summary_df["Norm_strain"] = (summary_df["Net_benefit"] - summary_df["Original_benefit"]) / summary_df["Net_benefit"]

# Counterfactual
summary_df["Equitable_cost"] = ((summary_df["OG_Cont"]/100*total_balance-summary_df["nfip_balance"])/summary_df["no_policies"])/100
summary_df["Solvent"] = ((summary_df["nfip_balance"] / summary_df["no_policies"]) * -1)/100

# Merge back to gdf_states for plotting
gdf_plot = gdf_states.merge(summary_df, on="STATEFP", how="left")

In [ ]:
# Panel metadata for maps a)-d), and shared map extent
plot_data = [
    ("Norm_re", "Premium-Relative Expected\nReinsurance Use", "a)", lambda v: f"{v:.1f}", "YlOrRd", None, ""),
    ("Norm_balance", "Premium-Relative Expected\nAnnual Contribution", "b)", lambda v: f"{v:.0f}", "RdYlBu", None, ""),
    ("Net_benefit", "State Benefit", "c)", lambda v: f"1e{int(np.round(np.log10(v)))}" if v > 0 else "0", "Blues",
     LogNorm(vmin=gdf_plot["Net_benefit"].replace(0, np.nan).min(), vmax=gdf_plot["Net_benefit"].max()), ""),
    ("Norm_strain", "National Strain", "d)", lambda v: f"{v:.1f}", "RdYlBu_r", None, ""),
]
extent = [-130, -65, 24, 50]
x_min, x_max, y_min, y_max = extent

In [ ]:
df_wide = pd.read_csv(f"Results/DTW_HC_simulation_clusters{test_case}.csv")
# Set 'state' as index so rows are states, columns are simulations
df_wide_indexed = df_wide.set_index("state")

# Convert each row to a list of cluster labels
cluster_records = df_wide_indexed.apply(lambda row: row.tolist(), axis=1).to_dict()

In [ ]:
# Assuming you already built cluster_records[state] = list of clusters from previous step
states = sorted(cluster_records.keys())
state_idx = {state: i for i, state in enumerate(states)}
n_states = len(states)
n_sims = len(next(iter(cluster_records.values())))

# Step 1: Build co-association matrix
coassoc_matrix = np.zeros((n_states, n_states))

for sim in range(n_sims):
    # Reconstruct clustering for this simulation
    sim_labels = {state: cluster_records[state][sim] for state in states}
    
    for i, state_i in enumerate(states):
        for j, state_j in enumerate(states):
            if sim_labels[state_i] == sim_labels[state_j]:
                coassoc_matrix[i, j] += 1

# Normalize to [0,1]
coassoc_matrix /= n_sims

# Step 2: Convert to dissimilarity matrix
dissimilarity = 1 - coassoc_matrix

# Step 3: Hierarchical clustering on dissimilarity
Z = linkage(dissimilarity, method='average')

# Choose number of consensus clusters
k = 3
final_labels = fcluster(Z, k, criterion='maxclust')

# Output DataFrame
cluster_df = pd.DataFrame({
    'state': states,
    'consensus_cluster': final_labels
})

In [ ]:
fig = plt.figure(figsize=(7, 11))
gs = fig.add_gridspec(4, 2, height_ratios=[1, 1, 1, 2], hspace=0.2, wspace=0.2)

# Panels a-d: rows 0-1
map_positions = [(0, 0), (0, 1), (1, 0), (1, 1)]
for i, ((row, col_idx), (col, title, label, fmt_fn, cmap, norm, cbar_label)) in enumerate(
        zip(map_positions, plot_data)):
    ax = fig.add_subplot(gs[row, col_idx])
    gdf_plot.plot(column=col, cmap=cmap, linewidth=0.1, ax=ax, norm=norm, legend=False)
    gdf_plot.boundary.plot(ax=ax, color="grey", linewidth=0.5)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.axis("off")
    ax.set_title(title, fontsize=11)
    ax.text(-0.06, 1.02, label, transform=ax.transAxes, size=12, weight='bold')
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm if norm else plt.Normalize(
        vmin=gdf_plot[col].min(), vmax=gdf_plot[col].max()))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, orientation='vertical', fraction=0.03, pad=0.02)
    cbar.set_label(cbar_label, fontsize=8)
    cbar.ax.tick_params(labelsize=7)

# Panel e: row 2, col 0
ax_e = fig.add_subplot(gs[2, 0])
pos = ax_e.get_position()
ax_e.set_position([pos.x0 + 0.04, pos.y0 + 0.02, pos.width - 0.03, pos.height - 0.03])
solvent_states = gdf_plot[gdf_plot["Solvent"] > 0].sort_values("Solvent", ascending=True)
ax_e.barh(solvent_states["STUSPS"], solvent_states["Solvent"], color="#eb5b3b", edgecolor="grey", linewidth=0.5)
ax_e.set_xlabel("Annual Cost Increase ($/HH)", fontsize=9)
ax_e.set_title("Annual Household Cost\nfor State Solvency", fontsize=11)
ax_e.tick_params(labelsize=8)
ax_e.spines["top"].set_visible(False)
ax_e.spines["right"].set_visible(False)
ax_e.text(-0.12, 1.02, "e)", transform=ax_e.transAxes, size=12, weight='bold')

# Panel f: row 2, col 1
ax_f = fig.add_subplot(gs[2, 1])
eq_min = gdf_plot["Solvent"].min()
eq_max = gdf_plot["Solvent"].max()
abs_max = max(abs(eq_min), abs(eq_max))
norm_f = TwoSlopeNorm(vmin=-abs_max, vcenter=0, vmax=abs_max)
gdf_plot.plot(column="Solvent", cmap="RdYlBu_r", linewidth=0.1, ax=ax_f, norm=norm_f, legend=False)
gdf_plot.boundary.plot(ax=ax_f, color="grey", linewidth=0.5)
ax_f.set_xlim(x_min, x_max)
ax_f.set_ylim(y_min, y_max)
ax_f.axis("off")
ax_f.set_title("Annual Household Cost\nfor Net Neutrality ($)", fontsize=11)
ax_f.text(-0.06, 1.02, "f)", transform=ax_f.transAxes, size=12, weight='bold')
sm_f = plt.cm.ScalarMappable(cmap="RdYlBu_r", norm=norm_f)
sm_f.set_array([])
cbar_f = fig.colorbar(sm_f, ax=ax_f, orientation='vertical', fraction=0.03, pad=0.02)
cbar_f.ax.tick_params(labelsize=7)

# Panel g: row 3, spans both columns
ax_g = fig.add_subplot(gs[3, :])
gdf_cluster = gdf_states.copy()
gdf_cluster = gdf_cluster.merge(cluster_df, left_on="STUSPS", right_on="state", how="left")
custom_cmap = ListedColormap(["#F4EEB5", "#C96A55", "#6299C3"], name="custom_ybr")
labels_g = ['Balanced', 'Imbalanced', 'Price-stabilized']
gdf_cluster.plot(column='consensus_cluster', cmap=custom_cmap, linewidth=0.5,
                 ax=ax_g, edgecolor='black', legend=False)
ax_g.set_xlim(x_min, x_max)
ax_g.set_ylim(y_min, y_max)
ax_g.axis("off")
ax_g.set_title("Unsupervised Clustering: State Pool Typologies", fontsize=11)
ax_g.text(-0.03, 1.02, "g)", transform=ax_g.transAxes, size=12, weight='bold')
unique_clusters = sorted(gdf_cluster["consensus_cluster"].dropna().unique())
cmap_g = plt.cm.get_cmap(custom_cmap, len(unique_clusters))
legend_elements = [
    Patch(facecolor=cmap_g(i), edgecolor='black', label=labels_g[i])
    for i, _ in enumerate(unique_clusters)
]
ax_g.legend(handles=legend_elements, loc="lower right", fontsize=7, frameon=False)

if save:
    plt.savefig(f"Plots/Fig2_{test_case}.pdf", dpi=500, bbox_inches='tight')
plt.show()